## Spatio-Temporal Graph Transformer Networks (ST-GTNs)

 **Spatio-Temporal Graph Transformer Networks (ST-GTNs)** are a class of deep learning architectures designed to forecast time series data situated on a graph, where multiple interconnected variables interact according to a spatial structure—such as sensors connected by roads (traffic flow), weather stations influenced by geography (weather), or nodes in an energy grid. Traditional models like RNNs/LSTMs are proficient in modeling temporal dependencies but falter at capturing spatial structures and struggle with long sequences, while CNNs/GCNs handle spatial patterns well but are limited in temporal scope, and Transformers excel at long-range dependencies by virtue of self-attention but are intrinsically designed for sequential (1D) rather than graph-structured (non-Euclidean) data. ST-GTNs address these limitations by combining the ability of Graph Neural Networks (GNNs) to model spatial dependencies with the Transformer’s power to capture long-term temporal relationships via self-attention mechanisms, thus providing a unified approach for learning from spatio-temporal data on graphs.

Let the sensor network be represented as a graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathbf{A})$, where:
*   $\mathcal{V}$ is the set of $N$ nodes (sensors).
*   $\mathcal{E}$ is the set of edges.
*   $\mathbf{A} \in \mathbb{R}^{N \times N}$ is the Adjacency Matrix representing spatial connectivity.

The input is a multivariate time series tensor:

$$ \mathbf{X} \in \mathbb{R}^{T \times N \times C} $$

*   $T$: Number of historical time steps (look-back window).
*   $N$: Number of nodes.
*   $C$: Number of features per node (e.g., speed, volume, occupancy).

**Goal:** Predict the future values for the next $T'$ time steps:

$$ \hat{\mathbf{Y}} \in \mathbb{R}^{T' \times N \times C} $$


The ST-GTN architecture for groundwater forecasting using time series inputs from weather variables, landuse, NDVI, and population density consists of three main stages: **Input Embedding**, **Spatio-Temporal Transformer Blocks**, and the **Output Head**.

***A. Input Embedding Layer***

For groundwater forecasting, the raw input consists of a tensor collecting biweekly weather features (such as precipitation, temperature, surface runoff, etc.), remotely sensed NDVI, static landuse information, and population density for each well location, all arranged according to a well network (graph).

These inputs are combined over the temporal dimension (biweekly sequence) and the spatial node dimension (wells) into a data tensor:
  $$ \mathbf{X} \in \mathbb{R}^{T \times N \times C} $$
where $T$ is number of time steps, $N$ the number of wells (nodes), and $C$ the number of input features (weather, NDVI, landuse, population).

(a)  **Feature Projection:**
    The input features for each node and time step are projected into a hidden space using a learnable linear transformation:
    $$ \mathbf{H}_{feat} = \text{Linear}(\mathbf{X}) $$

(b)  **Temporal Positional Encoding ($\mathbf{E}_{time}$):**
    To allow the model to understand the order of biweekly inputs, either a sinusoidal or learnable positional encoding is added for each time step $t$ (e.g., encoding that $t=0,\ldots,T-1$ refers to different periods in the year or history).

(c) **Spatial Node Embedding ($\mathbf{E}_{node}$):**
    Each well location (node) receives its own learnable embedding vector, capturing information about its spatial position and possible local characteristics not in the feature vector.

**Combined Input Embedding:**

$$ \mathbf{H}^{(0)} = \mathbf{H}_{feat} + \mathbf{E}_{time} + \mathbf{E}_{node} $$

with $\mathbf{H}^{(0)} \in \mathbb{R}^{T \times N \times d_{model}}$.

***B. Spatio-Temporal Transformer Blocks***

These are the model's core, stacked $L$ times. In groundwater applications, each block combines:
  - **Spatial Graph Attention**: Models how groundwater levels at each well relate to neighboring wells (connectivity defined by a spatial adjacency matrix, e.g., based on hydrogeological connectivity or physical distance).
  - **Temporal (Causal) Attention**: Models how a well's past (and related wells) influences its future groundwater levels over time.

The standard, sequential block performs:

(a) **Spatial Graph Attention Layer**  
For each biweekly time step $t$, attention is computed over all wells, but modulated by an adjacency mask derived from the connectivity of the groundwater network:

$$ Q = \mathbf{h}_t W_Q, \quad K = \mathbf{h}_t W_K, \quad V = \mathbf{h}_t W_V $$

$$ \text{Attention}(Q, K, V) = \text{softmax}\left( \frac{QK^T}{\sqrt{d_k}} + \mathbf{M}_{graph} \right) V $$

where $\mathbf{M}_{graph}$ is:

$$ \mathbf{M}_{graph}[i, j] = 
   \begin{cases} 
     0, & \text{if wells $i$ and $j$ are connected} \\
     -\infty, & \text{otherwise}
   \end{cases}
$$

This ensures information only passes between hydrologically or spatially connected wells.

(b) **Temporal Self-Attention Layer**

For each well (node $i$), attention is computed over its historical sequence, with causal masking to ensure only current and past biweekly time steps are visible (no leakage from the future):

$$ Q = \mathbf{h}_i W_Q^{temp}, \quad K = \mathbf{h}_i W_K^{temp}, \quad V = \mathbf{h}_i W_V^{temp} $$

$$ \mathbf{M}_{time}[t, t'] = 
   \begin{cases} 
     0, & t' \leq t \\
     -\infty, & t' > t
   \end{cases}
$$

$$ \text{TemporalAttention}(Q, K, V) = \text{softmax}\left( \frac{QK^T}{\sqrt{d_k}} + \mathbf{M}_{time} \right) V $$

***Feed-Forward Network (FFN) & Normalization***

Each attention sub-layer is wrapped with residual connection and layer normalization, then followed by a positionwise feed-forward network:

$$ \mathbf{H}' = \text{LayerNorm}\left( \mathbf{H} + \text{Attention}(\mathbf{H}) \right) $$

$$ \mathbf{H}^{(l)} = \text{LayerNorm}\left( \mathbf{H}' + \text{FFN}(\mathbf{H}') \right) $$

$$ \text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2 $$

***C. Output Head***

The final hidden representations, shaped $\mathbf{H}^{(L)} \in \mathbb{R}^{T \times N \times d_{model}}$, are used to predict groundwater levels for the forecast horizon (e.g., for $T'$ future biweekly steps) at each well.  
Commonly, the state at the final historical time step is projected to the forecast period:
$$ \hat{\mathbf{Y}} = \text{Linear}(\mathbf{H}^{(L)}_{\text{last}}) $$

Or, in sequence-to-sequence setups:

$$ \hat{\mathbf{Y}} = \text{Linear}(\text{Reshape}(\mathbf{H}^{(L)})) $$

$$ \hat{\mathbf{Y}} = \text{Linear}(\text{Reshape}(\mathbf{H}^{(L)})) $$

 $$ \hat{\mathbf{Y}} = \text{Linear}(\text{Reshape}(\mathbf{H}^{(L)})) $$


***Mathematical Summary of the Forward Pass***

Let $\mathcal{T}_l(\cdot)$ denote the Temporal Attention operation and $\mathcal{S}_l(\cdot)$ denote the Spatial Graph Attention operation at layer $l$.

(a)  **Input:** $\mathbf{X} \in \mathbb{R}^{T \times N \times C}$

(b)  **Embedding:** $\mathbf{H}^{(0)} = \text{Embed}(\mathbf{X})$

(c)  **For** $l = 1$ to $L$:
    *   **Spatial Step:** (Operates on Node dimension $N$)
        $$ \tilde{\mathbf{H}}^{(l)} = \text{LayerNorm}\left( \mathbf{H}^{(l-1)} + \mathcal{S}_l(\mathbf{H}^{(l-1)}, \mathbf{A}) \right) $$
    *   **Temporal Step:** (Operates on Time dimension $T$)
        $$ \mathbf{H}^{(l)} = \text{LayerNorm}\left( \tilde{\mathbf{H}}^{(l)} + \mathcal{T}_l(\tilde{\mathbf{H}}^{(l)}) \right) $$

(d)  **Output:** $\hat{\mathbf{Y}} = \text{Readout}(\mathbf{H}^{(L)})$



***Training Objective***

The model is trained by minimizing the difference between the predicted values $\hat{\mathbf{Y}}$ and the ground truth $\mathbf{Y}$. Common loss functions include:

**Mean Squared Error (MSE):**
$$ \mathcal{L}(\theta) = \frac{1}{T' \cdot N} \sum_{t=1}^{T'} \sum_{i=1}^{N} \| \mathbf{Y}_{t,i} - \hat{\mathbf{Y}}_{t,i} \|_2^2 $$

**Mean Absolute Error (MAE):**
$$ \mathcal{L}(\theta) = \frac{1}{T' \cdot N} \sum_{t=1}^{T'} \sum_{i=1}^{N} \| \mathbf{Y}_{t,i} - \hat{\mathbf{Y}}_{t,i} \| $$

Optimization is typically performed using **Adam** or **AdamW**.






![ST-GTN Architecture](ST_GTN-arcitect.png)

ST-GTNs are especially important for groundwater level forecasting because groundwater dynamics are inherently spatio-temporal: wells influence each other through hydrogeologic connectivity (captured via graph-masked spatial attention), while groundwater responses to drivers such as climate and pumping show delayed and long-range temporal effects (captured via self-attention over time). By embedding graph-structured historical signals into a shared latent space and stacking **Graph-Masked Spatial Attention** with **Causal Temporal Attention**, the model can learn *where* influences propagate across the well network and *when* they affect future levels. The main trade-off is that attention can be computationally expensive (scaling with both the number of nodes and the time window) and may require sufficient training data, but for complex groundwater systems this joint modeling can provide stronger forecasts than approaches that treat space or time in isolation.